# 03 — Pandas analysis

Same database as the SQL section. I load the six tables from SQLite rather than going back to the CSVs, so I'm not analysing a different copy of the data.

Budget and revenue of 0 stay in the tables. From question 2 onward, profit, ROI and anything about money uses only rows where both values are greater than 0.


In [1]:
import sqlite3
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

conn = sqlite3.connect("data/tmdb.db")
movies = pd.read_sql("SELECT * FROM movies", conn)
genres = pd.read_sql("SELECT * FROM genres", conn)
# rowid is the insert order, which is the order the genres appeared in the csv
movie_genres = pd.read_sql(
    "SELECT rowid, movie_id, genre_id FROM movie_genres ORDER BY rowid", conn
)
cast = pd.read_sql("SELECT * FROM cast", conn)
crew = pd.read_sql("SELECT * FROM crew", conn)
keywords = pd.read_sql("SELECT * FROM movie_keywords", conn)

movies["release_date"] = pd.to_datetime(movies["release_date"])
print(movies.shape, genres.shape, movie_genres.shape, cast.shape, crew.shape, keywords.shape)


(2503, 12) (19, 2) (6878, 3) (24902, 6) (7289, 6) (41681, 3)


## 1. How many unique movies, genres, actors and crew?


In [2]:
print("movies:       ", movies["movie_id"].nunique())
print("genres:       ", genres["genre_id"].nunique())
print("cast people:  ", cast["person_id"].nunique())
print("crew people:  ", crew["person_id"].nunique())


movies:        2503
genres:        19
cast people:   11300
crew people:   2848


2,503 movies, 19 genres, 11,300 distinct actors and 2,848 distinct crew members. I counted `person_id`, not the name string. Someone who is both an actor and a director is in both counts. That's what the question asks for.


## 2. Missing or zero budget and revenue


In [3]:
missing_money = movies[(movies["budget"] == 0) | (movies["revenue"] == 0)].copy()
print("budget is 0: ", int((movies["budget"] == 0).sum()))
print("revenue is 0:", int((movies["revenue"] == 0).sum()))
print("either is 0: ", len(missing_money))
print("both are 0:  ", int(((movies["budget"] == 0) & (movies["revenue"] == 0)).sum()))
missing_money[["title", "release_date", "budget", "revenue"]].sort_values("title")


budget is 0:  90
revenue is 0: 102
either is 0:  133
both are 0:   59


,title,release_date,budget,revenue
2360,365 Days,2020-02-07,0,9458590
2211,6 Underground,2019-12-10,150000000,0
158,8½,1963-02-14,0,0
1420,A Serbian Film,2010-01-06,0,0
1976,A Silent Voice: The Movie,2016-09-17,0,30819442
...,...,...,...,...
2088,When We First Met,2018-02-08,10000000,0
1228,Whisper of the Heart,1995-07-15,0,0
1529,Wolf Children,2012-07-21,0,55262158
2409,Zack Snyder's Justice League,2021-03-18,70000000,0


133 movies have a 0 in budget or revenue (or both). 59 of those are 0 in both columns. There are no nulls, only zeros.

I am treating 0 as missing, not as a real figure. For the rest of this notebook, and in the charts, profit and ROI only use movies where budget > 0 and revenue > 0. Ratings, counts and genre questions still include the zero-money rows, because the rating isn't missing just because the budget is.


## 3. Profit and ROI where both figures exist


In [4]:
valid = movies[(movies["budget"] > 0) & (movies["revenue"] > 0)].copy()
valid["profit"] = valid["revenue"] - valid["budget"]
valid["roi"] = valid["profit"] / valid["budget"]

print("movies with both figures:", len(valid))
print("median profit:", round(valid["profit"].median()))
print("median ROI:   ", round(valid["roi"].median(), 2))
print("share with profit > 0:", round((valid["profit"] > 0).mean(), 3))

valid[["title", "budget", "revenue", "profit", "roi"]].sort_values("profit", ascending=False)


movies with both figures: 2370
median profit: 88510698
median ROI:    2.43
share with profit > 0: 0.919


,title,budget,revenue,profit,roi
1136,Avatar,237000000,2923706026,2686706026,11.336312
1811,Avengers: Endgame,356000000,2799439100,2443439100,6.863593
217,Titanic,200000000,2264162353,2064162353,10.320812
1436,Avatar: The Way of Water,460000000,2334484620,1874484620,4.074967
1575,Star Wars: The Force Awakens,245000000,2068223624,1823223624,7.441729
...,...,...,...,...,...
2213,Red Notice,160000000,178143,-159821857,-0.998887
2412,The Killer,175000000,362113,-174637887,-0.997931
2304,The Tomorrow War,200000000,14400000,-185600000,-0.928000
2384,The Gray Man,200000000,454023,-199545977,-0.997730


2,370 movies have both numbers. Median profit is about $89 million and median ROI is about 2.4, so the typical film here a bit more than doubles its reported budget. The mean profit is higher, about $165 million, because the big titles pull it up. About 92% of these rows show a profit. I don't fully believe that rate. The file is full of films that were popular enough to land in this extract, and the revenue field is shaky for streaming titles. Avatar is the largest profit, about $2.69 billion.


## 4. Top 15 ROI, budget at least $1 million


In [5]:
roi_pool = valid[valid["budget"] >= 1_000_000].copy()
top_roi = roi_pool.nlargest(15, "roi")
top_roi[["title", "release_date", "budget", "revenue", "profit", "roi"]]


,title,release_date,budget,revenue,profit,roi
2191,Dragon Ball Super: Broly,2018-12-14,1000000,125002821,124002821,124.002821
152,Snow White and the Seven Dwarfs,1938-01-22,1488423,184925486,183437063,123.242561
1211,The Rocky Horror Picture Show,1975-08-14,1400000,171181400,169781400,121.272429
412,Rocky,1976-12-03,1000000,117253345,116253345,116.253345
300,Gone with the Wind,1939-12-15,4000000,402352579,398352579,99.588145
794,The Jungle Book,1967-10-18,4000000,378000000,374000000,93.500000
1001,Cinderella,1950-02-22,2900000,263600000,260700000,89.896552
78,Saw,2004-10-01,1200000,104045735,102845735,85.704779
1055,One Hundred and One Dalmatians,1961-01-25,3600000,303000000,299400000,83.166667
221,E.T. the Extra-Terrestrial,1982-06-11,10500000,797307407,786807407,74.934039


The top of this list is not the top of the revenue list. Dragon Ball Super: Broly, Snow White and the Seven Dwarfs, The Rocky Horror Picture Show and Rocky all return more than 100x, on budgets between $1 million and about $4 million. The $1 million floor is doing real work. Without it, a movie with a $7,000 budget and a modest gross would look like the best investment in history.

I would not pitch "make a $1 million film" from this. These are older films or films that became events. The median ROI above (about 2.4x) is the more honest number for a film that already has a normal budget.


## 5. Movies released per year


In [6]:
movies["year"] = movies["release_date"].dt.year
per_year = movies.groupby("year").size().rename("n_movies").reset_index()
peak = per_year.loc[per_year["n_movies"].idxmax()]
print("peak year:", int(peak["year"]), "with", int(peak["n_movies"]), "movies")
per_year.sort_values("n_movies", ascending=False).head(10)


peak year: 2016 with 121 movies


,year,n_movies
78,2016,121
75,2013,110
79,2017,109
76,2014,105
80,2018,102
81,2019,100
73,2011,97
77,2015,97
72,2010,90
71,2009,85


2016 has the most rows, 121 movies. 2013 through 2019 are all at 100 or more. The file runs from 1921 to 2026, but it is thick in the 2010s and thin before that. So "2016 was the biggest year for movies" is not a claim I would make. It is the year this extract contains the most titles.


## 6. Runtime outliers

IQR rule: anything below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR.


In [7]:
q1 = movies["runtime"].quantile(0.25)
q3 = movies["runtime"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print(f"Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}, fences={low:.1f} to {high:.1f}")

outliers = movies[(movies["runtime"] < low) | (movies["runtime"] > high)].copy()
print("outliers:", len(outliers))
print("shorter than the lower fence:", int((outliers["runtime"] < low).sum()))
print("longer than the upper fence:", int((outliers["runtime"] > high).sum()))
outliers[["title", "runtime", "release_date"]].sort_values("runtime")


Q1=99, Q3=126, IQR=27, fences=58.5 to 166.5
outliers: 60
shorter than the lower fence: 2
longer than the upper fence: 58


,title,runtime,release_date
2404,The Guardians of the Galaxy Holiday Special,45,2022-11-24
1641,The Garden of Words,46,2013-05-31
2370,Dune: Part Two,167,2024-02-27
583,Gangs of New York,168,2002-12-14
2154,It Chapter Two,169,2019-09-04
1595,Interstellar,169,2014-11-05
118,Pirates of the Caribbean: At World's End,169,2007-05-19
1306,The Hobbit: An Unexpected Journey,169,2012-12-12
344,Saving Private Ryan,169,1998-07-24
50,Scarface,170,1983-12-09


60 films fall outside the fences, which sit at about 59 and 167 minutes. The middle of the data is 99 to 126 minutes, median 111.

Only two films are short outliers: The Guardians of the Galaxy Holiday Special (45 min) and The Garden of Words (46 min). The other 58 are long. The longest is Zack Snyder's Justice League at 242 minutes, then Gone with the Wind at 238. "Outlier" here mostly means epic, not bad data. I would not delete them. A 3 hour film is a real film.


## 7. Primary genre and average popularity

There is no "order" column on movie_genres. I used the first genre row for each movie, which is the first one in the original csv (same as the smallest `rowid` after the load).


In [8]:
first_genre = movie_genres.drop_duplicates("movie_id", keep="first")
first_genre = first_genre.merge(genres, on="genre_id")
with_genre = movies.merge(first_genre[["movie_id", "genre_name"]], on="movie_id", how="left")
print("movies with no genre:", int(with_genre["genre_name"].isna().sum()))

primary_pop = (
    with_genre.dropna(subset=["genre_name"])
    .groupby("genre_name")["popularity"]
    .agg(avg_popularity="mean", n_movies="count")
    .sort_values("avg_popularity", ascending=False)
)
primary_pop.round(2)


movies with no genre: 4


,avg_popularity,n_movies
genre_name,,
Science Fiction,21.26,119
Music,18.32,10
Animation,16.81,139
Family,16.71,48
Action,16.39,474
War,15.47,23
Horror,14.85,223
Adventure,14.79,244
Fantasy,13.02,93


Science fiction as a *primary* genre has the highest average popularity, about 21, but that is only 119 films. Action is the primary genre on 474 movies and sits around 16. Comedy is primary on 361 and the average popularity is about 10, close to romance.

A few genres at the top of this table are tiny (music is 10 films, documentary is 1). I wouldn't rank those against drama. The comparison that holds up is among the big groups: action and adventure sit higher than comedy, drama and romance.


## 8. Top 10 actors by movie count, with their average rating


In [9]:
actor_ratings = (
    cast.merge(movies[["movie_id", "vote_average"]], on="movie_id")
    .groupby(["person_id", "actor_name"])
    .agg(n_movies=("movie_id", "nunique"), avg_rating=("vote_average", "mean"))
    .reset_index()
    .sort_values(["n_movies", "avg_rating"], ascending=[False, False])
    .head(10)
)
actor_ratings


,person_id,actor_name,n_movies,avg_rating
679,2231,Samuel L. Jackson,39,6.938692
106,287,Brad Pitt,38,7.257289
127,380,Robert De Niro,38,7.228316
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
1272,5293,Willem Dafoe,32,7.114781
413,1245,Scarlett Johansson,32,7.056344
2674,13240,Mark Wahlberg,32,6.630063
161,500,Tom Cruise,31,7.040613
64,192,Morgan Freeman,31,6.963742


Same names as the SQL count, with ratings next to them. Samuel L. Jackson leads on volume (39 films) with an average rating of about 6.94, which is basically the dataset average. Tom Hanks is a bit lower on count (33) and higher on rating (about 7.29). Mark Wahlberg is in 32 films but the average rating is about 6.63, the low end of this top 10.

So the prolific list and the "their films are well liked" list are not the same. If I were using this for casting I would look at both columns, not just the count.


## 9. Titles that appear more than once


In [10]:
repeat_titles = movies[movies["title"].duplicated(keep=False)].copy()
repeat_titles["year"] = repeat_titles["release_date"].dt.year
print("titles:", repeat_titles["title"].nunique(), "| rows:", len(repeat_titles))
repeat_titles[["title", "year", "movie_id"]].sort_values(["title", "year"])


titles: 49 | rows: 100


,title,year,movie_id
140,A Nightmare on Elm Street,1984,377
1164,A Nightmare on Elm Street,2010,23437
323,Aladdin,1992,812
2068,Aladdin,2019,420817
1042,Alice in Wonderland,1951,12092
...,...,...,...
1368,The Thing,2011,60935
346,Total Recall,1990,861
1381,Total Recall,2012,64635
2500,Transformers,1987,1368168


49 titles appear more than once (100 rows). A lot of these are remakes or a new version of an old film: Aladdin 1992 and 2019, Dune 1984 and 2021, Halloween three times, Robin Hood three times.

A few look wrong rather than like remakes. Fargo is in here as 1996 and again as 2011-03-08, and Date Night as 1995 and 2010 with the same month and day. Those share a title and a calendar day but the movie ids differ, so they survived the duplicate-row drop in the cleaning notebook. I left them. Collapsing them automatically would also collapse real remakes like Dune.


## 10. Popularity and vote count that don't follow the usual pattern

I scaled popularity, scaled `log(1 + vote_count)`, and fit a straight line. The residual is actual votes minus what the line predicted from popularity. A large negative residual means the film is popular relative to how many votes it has. A large positive residual means the opposite: lots of votes, popularity score not keeping up.


In [11]:
pop_z = (movies["popularity"] - movies["popularity"].mean()) / movies["popularity"].std()
log_votes = np.log1p(movies["vote_count"])
votes_z = (log_votes - log_votes.mean()) / log_votes.std()

slope, intercept = np.polyfit(pop_z, votes_z, 1)
movies = movies.copy()
movies["residual"] = votes_z - (slope * pop_z + intercept)

cols = ["title", "release_date", "popularity", "vote_count", "residual"]
print("fewer votes than the popularity score suggests")
display(movies.nsmallest(10, "residual")[cols])
print("more votes than the popularity score suggests")
display(movies.nlargest(10, "residual")[cols])


fewer votes than the popularity score suggests


,title,release_date,popularity,vote_count,residual
2495,Disclosure Day,2026-06-10,394.3346,2328,-7.780526
2497,Obsession,2026-05-13,364.8722,4203,-6.314570
2479,Backrooms,2026-05-27,258.4743,2483,-5.340068
2345,Spider-Man: No Way Home,2021-12-15,438.4757,22429,-4.860016
2501,The SpongeBob Movie: Sponge Out of Water,2000-01-28,11.3857,475,-3.775973
2444,Mortal Kombat II,2026-05-06,99.5231,2260,-2.760089
2499,Date Night,1995-04-08,8.2930,1051,-2.434529
2496,Lee Cronin's The Mummy,2026-04-15,79.4601,2430,-2.297386
2490,The Super Mario Galaxy Movie,2026-04-01,97.1210,3343,-2.083029
2369,Project Hail Mary,2026-03-15,162.7023,6750,-2.069299


more votes than the popularity score suggests


,title,release_date,popularity,vote_count,residual
1540,Guardians of the Galaxy,2014-07-30,18.9524,29995,2.824859
1794,Deadpool,2016-02-09,29.3387,33003,2.801536
1180,Inception,2010-07-15,55.8497,39727,2.646968
1401,The Hunger Games,2012-03-12,12.2061,23506,2.544850
3,Forrest Gump,1994-06-23,35.6479,30101,2.543551
1136,Avatar,2009-12-16,48.1996,34348,2.542157
2156,Joker,2019-10-01,30.2147,28002,2.519543
270,Pulp Fiction,1994-09-10,42.0350,30542,2.457366
1010,Shutter Island,2010-02-14,27.5156,26135,2.453861
1805,Suicide Squad,2016-08-03,13.7141,22256,2.430158


The negative side is mostly very new releases. Disclosure Day, Obsession and Backrooms are all dated 2026 and already have high popularity with only a couple of thousand votes. Spider-Man: No Way Home also lands here, which is different: it has over 20,000 votes, but its popularity score is so high (about 438) that the line still expected more.

The positive side is older, heavily voted films whose popularity number has cooled off: Guardians of the Galaxy, Deadpool, Inception, Forrest Gump, Pulp Fiction. Popularity on TMDB moves. Vote count doesn't go backwards. I would not treat a mismatch between the two as an error in the row.
